In [11]:
import tensorflow as tf
import tensorflow_probability as tfp 
import tensorflow_addons as tfa
from tensorflow import keras
from keras import mixed_precision

from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import numpy as np
import pandas as pd
import functions
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_curve, roc_auc_score, f1_score

import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [12]:
# 1. TensorFlow version
print("TF version:", tf.__version__)

# 2. Is this a GPU build?
print("Built with CUDA:", tf.test.is_built_with_cuda())

# 3. Any GPUs visible?
print("Physical GPUs:", tf.config.list_physical_devices('GPU'))

# 4. Full build_info dict
info = tf.sysconfig.get_build_info()
print("Build info keys:", info.keys())
tf.config.list_physical_devices()

TF version: 2.10.0
Built with CUDA: True
Physical GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Build info keys: odict_keys(['cpu_compiler', 'cuda_compute_capabilities', 'cuda_version', 'cudart_dll_name', 'cudnn_dll_name', 'cudnn_version', 'is_cuda_build', 'is_rocm_build', 'is_tensorrt_build', 'msvcp_dll_names', 'nvcuda_dll_name'])


[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'),
 PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [13]:
class stacked_lstm_encoder(keras.layers.Layer):
    def __init__(self, vec_size = 10, nominals_embeddings={}, hidden_units=64, dropout_rate=0.2, kernel_regularizer = None, **kwargs):
        super().__init__(**kwargs)
        self.latent_output_shape = vec_size
        self.lstm1 = keras.layers.LSTM(hidden_units, 
                                       return_sequences=True, 
                                       kernel_regularizer=kernel_regularizer,
                                       kernel_initializer=tf.keras.initializers.GlorotUniform(),
                                       recurrent_initializer=tf.keras.initializers.Orthogonal())
        self.layernorm1 = keras.layers.LayerNormalization()
        self.layernorm2 = keras.layers.LayerNormalization()
        self.hidden_dropout = keras.layers.Dropout(dropout_rate)
        self.skip_proj = keras.layers.Dense(self.latent_output_shape, activation="linear")
        self.last_proj = keras.layers.Dense(self.latent_output_shape, activation="linear")

        self.lstm2 = keras.layers.LSTM(self.latent_output_shape, 
                                       return_sequences=False,
                                       activation='tanh', 
                                       kernel_regularizer=kernel_regularizer,
                                       kernel_initializer=tf.keras.initializers.GlorotUniform(),
                                       recurrent_initializer=tf.keras.initializers.Orthogonal())
        self.global_pool = keras.layers.GlobalAveragePooling1D()


    def build(self, input_shape):
        self.n_features = input_shape[-1]
        return super().build(input_shape)
    
    def call(self, inputs, training=False):
        # (batch [number of windows], windows length, features_dim)
        hidden = self.lstm1(inputs, training=training)
        hidden = self.layernorm1(hidden)
        hidden = self.hidden_dropout(hidden, training=training)
        latent_reps = self.lstm2(hidden, training=training)
        latent_reps = self.layernorm2(latent_reps)
        return latent_reps 


In [14]:
class lstm_decoder(keras.layers.Layer):
    def __init__(self, dimensions, nominals_embeddings={}, hidden_units=64, dropout_rate=0.2, kernel_regularizer = None, **kwargs):
        super().__init__(**kwargs)
        self.window_length, self.feature_dim  = dimensions
        self.repeat_layer = keras.layers.RepeatVector(self.window_length)

        self.input_proj = keras.layers.TimeDistributed(keras.layers.Dense(hidden_units, activation='tanh',
                                                                            kernel_regularizer=kernel_regularizer,
                                                                            kernel_initializer=tf.keras.initializers.GlorotUniform()))
        self.lstm = keras.layers.LSTM(hidden_units, 
                                      return_sequences=True, 
                                      kernel_regularizer=kernel_regularizer,
                                      kernel_initializer=tf.keras.initializers.GlorotUniform(),
                                      recurrent_initializer=tf.keras.initializers.Orthogonal())
        self.out = keras.layers.TimeDistributed(keras.layers.Dense(self.feature_dim,
                                                                   kernel_regularizer=kernel_regularizer,
                                                                   activation='linear',
                                                                   kernel_initializer=tf.keras.initializers.GlorotUniform()))        
        # Consider linear outputs for for real valued TS  
        # self.out = keras.layers.TimeDistributed(keras.layers.Dense(self.feature_dim, kernel_regularizer=kernel_regularizer, activation='linear'))   

        self.initial_h = keras.layers.Dense(hidden_units, activation='tanh',
                                            kernel_regularizer=kernel_regularizer,
                                            kernel_initializer=tf.keras.initializers.GlorotUniform())
        self.initial_c = keras.layers.Dense(hidden_units, activation='linear',
                                            kernel_regularizer=kernel_regularizer,
                                            kernel_initializer=tf.keras.initializers.GlorotUniform())
        
        self.latent_to_seq = keras.layers.Dense(self.window_length * self.feature_dim,
                                                activation='linear',
                                                kernel_regularizer=kernel_regularizer,
                                                kernel_initializer=tf.keras.initializers.GlorotUniform())

    
    def call(self, latent_reps, training=False):
        # initial states
        h0 = self.initial_h(latent_reps, training=training)
        c0 = self.initial_c(latent_reps, training=training)
        x = self.repeat_layer(latent_reps) 
        x = self.lstm(x, initial_state=[h0,c0], training=training)
        return self.out(x, training=training)

In [15]:
class LSTMAE_CBCL:
    def __init__(self, dimensions, latent_vec_size=10, weights=[0.2, 2, 2], lstm_units=64, dropout_rate=0.2, kernel_regularizer=None, num_clusters=5, opt_lr=0.001, ema=0.1, clip_val=1):
        self.encoder = stacked_lstm_encoder(latent_vec_size, hidden_units=lstm_units, dropout_rate=dropout_rate, kernel_regularizer=kernel_regularizer)
        self.decoder = lstm_decoder(dimensions, hidden_units=lstm_units, dropout_rate=dropout_rate, kernel_regularizer=kernel_regularizer)
        self.num_clusters = num_clusters
        self.latent_dim = latent_vec_size
        self.centroid_shift_momentum = ema #[0.1-0.5]
        self.centroids = None
        self.alpha, self.lmda, self.gamma = weights
        self.mse = keras.losses.MeanSquaredError()
        self.optimizer = keras.optimizers.Adam(learning_rate=opt_lr)
        self.reconstruction_opt = keras.optimizers.Adam(learning_rate=opt_lr)
        self.cbcl_opt = keras.optimizers.SGD(learning_rate=opt_lr)
        self.encoder_weights = None
        self.decoder_weights = None
        self.centroids_backup = None
        self.clip_val = clip_val
        # ensure encoder/decoder variables exist
        dummy = tf.zeros([1, dimensions[0], dimensions[1]], dtype=tf.float32)
        _ = self.encoder(dummy)                     # builds encoder weights
        _ = self.decoder(self.encoder(dummy))       # builds decoder weights
        self.mse_weight = 1
        self.cbcl_weight = 1
        
        # initialize centroids to zeros so decorators that read them don't break
        if self.centroids is None:
            init = tf.zeros([self.num_clusters, self.latent_dim], dtype=tf.float32)
            self.centroids = tf.Variable(init, trainable=False, dtype=tf.float32)
            self.initialized = False
        
        self._input_signature_hard = [tf.TensorSpec(shape=[None, dimensions[0], dimensions[1]], dtype=tf.float32)]
        self.hard_train_step = tf.function(self.hard_train_step, input_signature=self._input_signature_hard)
        self.reconstruction_train_step=tf.function(self.reconstruction_train_step, input_signature=self._input_signature_hard)
        self.cbcl_train_step=tf.function(self.cbcl_train_step, input_signature=self._input_signature_hard)
            
    def ae_reconstruct(self, batch):
        Z = self.encode(batch)
        batch_reconstructed = self.decode(Z)
        return batch_reconstructed
    
    def encode(self, batch):
        return self.encoder(batch, training=False)
    
    def decode(self, batch):
        return self.decoder(batch, training=False)
    
    def save_curr_weights(self):
        self.encoder_weights = self.encoder.get_weights()
        self.decoder_weights = self.decoder.get_weights()
        self.centroids_backup = self.centroids.read_value().numpy()

    def load_stored_weights(self):
        self.encoder.set_weights(self.encoder_weights)
        self.decoder.set_weights(self.decoder_weights)
        self.centroids.assign(tf.convert_to_tensor(self.centroids_backup, dtype=self.centroids.dtype))
        
    def compute_batch_centroids(self, latents, assignments):
        """
        Performs Kmeans on the current batch 
        
        latents: [B, d] TF tensor  
        assignments: [B] int32 TF tensor of latent's cluster assignment, from [0, num_cluster)  
        returns: batch_centroids [K, d], counts [K] (float32)  
        """
        sum_per_cluster = tf.math.unsorted_segment_sum(latents, assignments, self.num_clusters)
        counts = tf.math.unsorted_segment_sum(tf.ones_like(assignments, dtype=latents.dtype), assignments, self.num_clusters)
        counts_nonzeros = tf.reshape(tf.maximum(counts, tf.cast(1.0, dtype=latents.dtype)), (-1, 1))
        batch_centroids = sum_per_cluster / counts_nonzeros
        return batch_centroids, tf.cast(counts, tf.float32)
    
    def compute_initial_centroids(self, train_ds):
        train_batches = train_ds.shuffle(train_ds.cardinality()).batch(2048).prefetch(tf.data.AUTOTUNE)
        kmeans = KMeans(n_clusters=self.num_clusters, n_init='auto', init='k-means++', random_state=42)
        all_latents = []
        for batch in train_batches:
            lat_ = self.encode(batch).numpy()
            all_latents.append(lat_)
        
        kmeans.fit(np.concatenate(all_latents))
                
        init_centers = tf.convert_to_tensor(kmeans.cluster_centers_.astype(np.float32), dtype=tf.float32)
        self.centroids.assign(tf.convert_to_tensor(init_centers, dtype=self.centroids.dtype))
        self.initialized=True
    
    @tf.function
    def pretrain_ae(self, batch):
        with tf.GradientTape() as tape:
            full_batch_latents = self.encoder(batch, training=True)
            full_batch_reconstructed = self.decoder(full_batch_latents, training=True)
            loss_reconstruction = tf.reduce_mean(self.mse(batch, full_batch_reconstructed))
        weights = self.encoder.trainable_weights + self.decoder.trainable_weights
        gradients = tape.gradient(loss_reconstruction, weights)
        gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
        grads_and_vars = [(g,w) for g, w in zip(gradients, weights) if g is not None]
        self.optimizer.apply_gradients(grads_and_vars)
    
    @tf.function
    def hard_assignments(self, latents):
        centroids = tf.cast(self.centroids.read_value(), tf.float32)
        dists = tf.reduce_sum(tf.square(tf.expand_dims(latents, 1) - tf.expand_dims(centroids, 0)),
                              axis=2)
        assignments = tf.cast(tf.argmin(dists, axis=1), tf.int32)
        
        #soft_probs = tf.nn.softmax(-dists, axis=1)
        #max_confidence = tf.reduce_max(probs, axis=1)
        #confidence_mask = max_confidence >= tf.cast(0.75, max_confidence.dtype)
        
        # One hot row vectors, where indices are the sample's cluster assignment
        onehot = tf.one_hot(assignments, depth=self.num_clusters, dtype=tf.float32)
        #onehot = tf.where(tf.expand_dims(confident_mask, -1), onehot, tf.zeros_like(onehot))
        
        return assignments, onehot
    
    @tf.function
    def hard_contrastive_loss(self, latents, assignments):

        Z, Z_prime = tf.split(latents, num_or_size_splits=2, axis=0)
        l2_sq = tf.reduce_sum(tf.square(Z-Z_prime), axis=1)
        pairwise_l2 = tf.sqrt(l2_sq + 1e-9)
        
        assignments_Z, assignments_Zprime = tf.split(assignments, num_or_size_splits=2,axis=0)
        same = tf.equal(assignments_Z, assignments_Zprime)
        diff = tf.logical_not(same)
        
        pos_terms = tf.where(same, l2_sq, tf.zeros_like(l2_sq))
        neg_terms = tf.where(diff, tf.square(tf.maximum(self.lmda - pairwise_l2, 0.0)), tf.zeros_like(pairwise_l2))
        
        total = pos_terms + neg_terms
        denom = tf.cast(tf.maximum(tf.shape(total)[0], 1), total.dtype)

        return tf.reduce_sum(total) / denom

    @tf.function   
    def hard_centroid_loss(self, batch_centroids):
        K = tf.shape(batch_centroids)[0]
        
        def compute():
            dp = tf.matmul(batch_centroids, batch_centroids, transpose_b=True)
            ss = tf.reduce_sum(tf.square(batch_centroids), axis=1, keepdims=True)
            centroid_distances = tf.sqrt(tf.maximum(ss + tf.transpose(ss) - 2.0 * dp, 1e-12))
            mask = tf.logical_not(tf.eye(K, dtype=tf.bool))
            masked = tf.boolean_mask(centroid_distances, mask)
            avg = tf.reduce_mean(masked)
            return tf.square(tf.maximum(tf.cast(self.gamma, tf.float32) - avg, 0.0))
        return tf.cond(K < 2, lambda: tf.constant(0.0, dtype=batch_centroids.dtype), compute)
        
    def hard_train_step(self, batch):
        batch = tf.cast(batch, tf.float32)
        B = tf.shape(batch)[0]
        N = B // 2
        
        # Force batch even for splitting
        batch = batch[:N*2]
        weights = self.encoder.trainable_weights + self.decoder.trainable_weights
        
        with tf.GradientTape(persistent=True) as tape:
            full_latents = tf.cast(self.encoder(batch, training=True), tf.float32) # [B, latent_dims]
            full_reconstructed = tf.cast(self.decoder(full_latents, training=True), tf.float32) # [B, T, O]
            loss_reconstruction = self.mse(batch, full_reconstructed) # Scalar loss
            assignments, onehot_assignments = self.hard_assignments(full_latents)
            loss_contrast = self.hard_contrastive_loss(full_latents, assignments) # Scalar loss
            curr_batch_centroids, counts = self.compute_batch_centroids(full_latents, assignments)
            
            loss_centroid = self.hard_centroid_loss(curr_batch_centroids) # Scalar loss
            total_loss = (1.0-self.alpha) * tf.cast(loss_reconstruction, tf.float32) + self.alpha * (tf.cast(loss_contrast, tf.float32) + tf.cast(loss_centroid, tf.float32))

        grads_recon = tape.gradient(loss_reconstruction, weights)
        grads_contrast = tape.gradient(loss_contrast, weights)
        grads_centroid = tape.gradient(loss_centroid, weights)
        grads_total_from_losses = tape.gradient(total_loss, weights)  # should equal (1-alpha)*g_recon + alpha*(g_contrast+g_centroid) in linear case

        # Helper: replace None grads with zero tensors of appropriate shape so norms work
        zeroed = []
        for g, w in zip(grads_total_from_losses, weights):
            if g is None:
                zeroed.append(tf.zeros_like(w))
            else:
                zeroed.append(g)

        def _zero_none(grads, weights):
            out = []
            for g, w in zip(grads, weights):
                out.append(g if g is not None else tf.zeros_like(w))
            return out

        ########## Gradient Debugging ###################################################
        grads_recon = _zero_none(grads_recon, weights)
        grads_contrast = _zero_none(grads_contrast, weights)
        grads_centroid = _zero_none(grads_centroid, weights)
        # Compose the combined gradient from components to check equality
        composed_grads = []
        for gr, gc, gce in zip(grads_recon, grads_contrast, grads_centroid):
            # scaling same as total_loss: (1-alpha) * recon + alpha * (contrast + centroid)
            composed_grads.append((1.0 - self.alpha) * gr + self.alpha * (gc + gce))
        # Global norms for each set
        def global_norm(grads):
            sq = [tf.reduce_sum(tf.square(g)) for g in grads]
            return tf.sqrt(tf.add_n(sq) + 1e-12)
        norm_recon = global_norm(grads_recon)
        norm_contrast = global_norm(grads_contrast)
        norm_centroid = global_norm(grads_centroid)

        # tf.print("GRAD-NORMS: recon:", norm_recon, "contrast:", norm_contrast, "centroid:", norm_centroid, '\n')
        #################################################################################
        
        gradients = tape.gradient(total_loss, weights)
        gradients, _global_norm_before_clipping = tf.clip_by_global_norm(gradients, self.clip_val)
        grads_and_vars = [(g,w) for g, w in zip(gradients, weights) if g is not None]
        self.optimizer.apply_gradients(grads_and_vars)
        del tape
        
        ########### EMA #################       
        old = self.centroids.read_value()
        mask = counts > 1e-6
        mask_exp = tf.expand_dims(mask, -1)
        m = tf.cast(self.centroid_shift_momentum, old.dtype)
        new_centroids = (1.0 - m) * old + m * tf.stop_gradient(curr_batch_centroids)
        new_centroids = tf.where(mask_exp, new_centroids, old)
        self.centroids.assign(new_centroids)
        
        return (1.0-self.alpha) * loss_reconstruction, self.alpha *loss_contrast, self.alpha * loss_centroid, total_loss
    
    def visualize_clustering_with_GMM(self, train_latents:np.ndarray, test_latents:np.ndarray, test_labels, radius_quantile=.95):
        def compute_ellipsoid_surface(center, cov, u_res=30, v_res=30):
            # center: (3,), cov: (3,3)
            vals, vecs = np.linalg.eigh(cov)
            vals = np.clip(vals, 1e-8, None)
            radii = np.sqrt(vals)

            u = np.linspace(0, 2 * np.pi, u_res)
            v = np.linspace(0, np.pi, v_res)
            u, v = np.meshgrid(u, v)

            xs = np.cos(u) * np.sin(v)
            ys = np.sin(u) * np.sin(v)
            zs = np.cos(v)

            pts = np.stack([xs.ravel(), ys.ravel(), zs.ravel()], axis=0)  # (3, M)
            transform = vecs @ np.diag(radii)
            ell_pts = transform @ pts
            ell_pts = ell_pts.T.reshape(u_res, v_res, 3)
            x = ell_pts[:, :, 0] + center[0]
            y = ell_pts[:, :, 1] + center[1]
            z = ell_pts[:, :, 2] + center[2]
            return x, y, z
       
        train_latents = np.vstack(train_latents)
        test_latents = np.vstack(test_latents)
        normal_mask = (test_labels.numpy() == 0)
        anom_mask = (test_labels.numpy()  == 1)
        
        gmm = GaussianMixture(n_components=5, covariance_type='full', random_state=42, n_init=10, reg_covar=1e-5)
        tsne = TSNE(n_components=3, random_state=42, perplexity=50, init='pca')

        gmm.fit(train_latents)
        self.pca.fit(train_latents)    
        self.umap.fit(train_latents) 
        umap_test_proj = self.umap.transform(test_latents)
        umap_train_proj = self.umap.transform(train_latents)
        print(self.pca.explained_variance_)
        
        pca_test_proj = self.pca.transform(test_latents)
        pca_gmm_means_proj = self.pca.transform(gmm.means_)
        pca_train_proj = self.pca.transform(train_latents)
        
        rng = np.random.RandomState(42)
        comp_samples = [] 
        for k in range(gmm.n_components):
            mean = gmm.means_[k]
            cov = gmm.covariances_[k]
            cov = cov + np.eye(cov.shape[0]) * 1e-8
            samples = rng.multivariate_normal(mean, cov, size=2000)
            comp_samples.append(samples)

        # Build concatenated array for joint t-SNE embedding:
        # [test_points, gmm.means_, comp1_samples, comp2_samples, ...]
        concat_parts = [train_latents, test_latents, gmm.means_] + comp_samples
        X_concat = np.vstack(concat_parts)
        tsne_proj = tsne.fit_transform(X_concat)
            
        # split back ()
        idx = 0
        N_test = test_latents.shape[0]
        N_train = train_latents.shape[0]
        embedded_train = tsne_proj[idx: idx + N_train]; idx += N_train
        embedded_test = tsne_proj[idx: idx + N_test]; idx += N_test
        N_means = gmm.means_.shape[0]
        embedded_means = tsne_proj[idx: idx + N_means]; idx += N_means
        embedded_comp_samples = []
        for k in range(N_means):
            ns = comp_samples[k].shape[0]
            embedded_comp_samples.append(tsne_proj[idx: idx + ns])
            idx += ns
        
        fig = make_subplots(rows=1, cols=3,
                        specs=[[{'type':'scene'}, {'type':'scene'},  {'type':'scene'}]],
                        subplot_titles=['PCA (3D)', 'UMAP (3D)', 'TSNE (3D)'])

        
        ##################### TRAIN latents ##########################
        
        # PCA 
        fig.add_trace(go.Scatter3d(
            x=pca_train_proj[:,0], y=pca_train_proj[:,1], z=pca_train_proj[:,2],
            mode='markers',
            marker=dict(size=2, color='blue', opacity=0.1),
            name='Train projections'
        ), row=1, col=1)
        
        # GMM means in PCA subplot (row=1,col=1)
        fig.add_trace(go.Scatter3d(
            x=pca_gmm_means_proj[:,0], y=pca_gmm_means_proj[:,1], z=pca_gmm_means_proj[:,2],
            mode='markers+text',
            marker=dict(size=8, symbol='x'),
            text=[f'comp {i}' for i in range(N_means)],
            textposition='top center',
            name='GMM means (PCA)'
        ), row=1, col=1)
        for k, samples in enumerate(comp_samples):
            samples_pca = self.pca.transform(samples)[:, :3]
            center = samples_pca.mean(axis=0)
            cov_pca = np.cov(samples_pca.T)
            x_s, y_s, z_s = compute_ellipsoid_surface(center, cov_pca, u_res=24, v_res=24)
            fig.add_trace(go.Surface(x=x_s, y=y_s, z=z_s, opacity=0.5, showscale=False, name=f'PCA cov comp {k}'),
                        row=1, col=1)
        
        # UMAP subplot (row=1,col=2)
        fig.add_trace(go.Scatter3d(
            x=umap_train_proj[:,0], y=umap_train_proj[:,1], z=umap_train_proj[:,2],
            mode='markers',
            marker=dict(size=2, color='blue', opacity=0.1),
            name='Train latents'
        ), row=1, col=2)
        
         # ---------- t-SNE subplot (col=2) ----------
        fig.add_trace(go.Scatter3d(
            x=embedded_train[:, 0], y=embedded_train[:, 1], z=embedded_train[:, 2],
            mode='markers',
            marker=dict(size=2, color='blue'),
            name='Train latents'
        ), row=1, col=3)

        # GMM means in embedded t-SNE space
        fig.add_trace(go.Scatter3d(
            x=embedded_means[:, 0], y=embedded_means[:, 1], z=embedded_means[:, 2],
            mode='markers+text',
            marker=dict(size=8, symbol='x'),
            text=[f'comp {i}' for i in range(N_means)],
            textposition='top center',
            name='GMM means (t-SNE)'
        ), row=1, col=3)

        # t-SNE ellipsoids from embedded component samples
        for k, samples_emb in enumerate(embedded_comp_samples):
            center = samples_emb.mean(axis=0)
            cov_emb = np.cov(samples_emb.T)
            x_s, y_s, z_s = compute_ellipsoid_surface(center, cov_emb, u_res=24, v_res=24)
            fig.add_trace(go.Surface(x=x_s, y=y_s, z=z_s, opacity=0.3, showscale=False, name=f'tSNE cov comp {k}'),
                        row=1, col=3)
        
        ############## TEST LATENTS ####################
        fig.add_trace(go.Scatter3d(
            x=pca_test_proj[normal_mask,0], y=pca_test_proj[normal_mask,1], z=pca_test_proj[normal_mask,2],
            mode='markers',
            marker=dict(size=2, color='green'),
            name='normal (0)'
        ), row=1, col=1)
        fig.add_trace(go.Scatter3d(
            x=pca_test_proj[anom_mask,0], y=pca_test_proj[anom_mask,1], z=pca_test_proj[anom_mask,2],
            mode='markers',
            marker=dict(size=3, color='red', symbol='diamond'),
            name='anomaly (1)'
        ), row=1, col=1)
            
        fig.add_trace(go.Scatter3d(
            x=embedded_test[normal_mask, 0], y=embedded_test[normal_mask, 1], z=embedded_test[normal_mask, 2],
            mode='markers',
            marker=dict(size=2, color='green'),
            name='normal (0) (t-SNE)'
        ), row=1, col=3)

        fig.add_trace(go.Scatter3d(
            x=embedded_test[anom_mask, 0], y=embedded_test[anom_mask, 1], z=embedded_test[anom_mask, 2],
            mode='markers',
            marker=dict(size=3, color='red', symbol='diamond'),
            name='anomaly (1) (t-SNE)'
        ), row=1, col=3)

        # UMAP subplot (row=1,col=2)
        fig.add_trace(go.Scatter3d(
            x=umap_test_proj[normal_mask,0], y=umap_test_proj[normal_mask,1], z=umap_test_proj[normal_mask,2],
            mode='markers',
            marker=dict(size=2, color='green'),
            name='normal (0)'
        ), row=1, col=2)
        fig.add_trace(go.Scatter3d(
            x=umap_test_proj[anom_mask,0], y=umap_test_proj[anom_mask,1], z=umap_test_proj[anom_mask,2],
            mode='markers',
            marker=dict(size=3, color='red', symbol='diamond'),
            name='anomaly (1)'
        ), row=1, col=2)
        
        fig.update_layout(height=700, width=1400,
                        scene=dict(xaxis_title='PC1', yaxis_title='PC2', zaxis_title='PC3'),
                        scene2=dict(xaxis_title='UMAP-1', yaxis_title='UMAP-2', zaxis_title='UMAP-3'),
                        scene3=dict(xaxis_title='TSNE-1', yaxis_title='TSNE-2', zaxis_title='TSNE-3'),
                        title='Test latents with GMM means and covariance ellipsoids')
        fig.show()
    
    ############ EXPERIMENTAL ####################
    def reconstruction_train_step(self, batch):
        weights = self.encoder.trainable_weights + self.decoder.trainable_weights
        
        with tf.GradientTape(persistent=True) as mse_tape:
            full_latents = tf.cast(self.encoder(batch, training=True), tf.float32) # [B, latent_dims]
            full_reconstructed = tf.cast(self.decoder(full_latents, training=True), tf.float32) # [B, T, O]
            loss_reconstruction = self.mse(batch, full_reconstructed) # Scalar loss
            loss_reconstruction = self.mse_weight * loss_reconstruction
        
        gradients = mse_tape.gradient(loss_reconstruction, weights)
        gradients, _global_norm_before_clipping = tf.clip_by_global_norm(gradients, self.clip_val)
        grads_and_vars = [(g,w) for g, w in zip(gradients, weights) if g is not None]
        self.optimizer.apply_gradients(grads_and_vars)
        del mse_tape
        return loss_reconstruction
    
    def cbcl_train_step(self, batch):
        batch = tf.cast(batch, tf.float32)
        B = tf.shape(batch)[0]
        N = B // 2
        
        # Force batch even for splitting
        batch = batch[:N*2]
        weights = self.encoder.trainable_weights + self.decoder.trainable_weights
        
        with tf.GradientTape(persistent=True) as cbcl_tape:
            full_latents = tf.cast(self.encoder(batch, training=True), tf.float32) # [B, latent_dims]
            assignments, onehot_assignments = self.hard_assignments(full_latents)
            loss_contrast = self.hard_contrastive_loss(full_latents, assignments) # Scalar loss
            curr_batch_centroids, counts = self.compute_batch_centroids(full_latents, assignments)
            loss_centroid = self.hard_centroid_loss(curr_batch_centroids) # Scalar loss
            
            total_loss = self.cbcl_weight*(tf.cast(loss_contrast, tf.float32) + tf.cast(loss_centroid, tf.float32))

        grads_contrast = cbcl_tape.gradient(loss_contrast, weights)
        grads_centroid = cbcl_tape.gradient(loss_centroid, weights)
        grads_total_from_losses = cbcl_tape.gradient(total_loss, weights)  # should equal (1-alpha)*g_recon + alpha*(g_contrast+g_centroid) in linear case

        # Helper: replace None grads with zero tensors of appropriate shape so norms work
        zeroed = []
        for g, w in zip(grads_total_from_losses, weights):
            if g is None:
                zeroed.append(tf.zeros_like(w))
            else:
                zeroed.append(g)

        def _zero_none(grads, weights):
            out = []
            for g, w in zip(grads, weights):
                out.append(g if g is not None else tf.zeros_like(w))
            return out

        ########## Gradient Debugging ###################################################
        grads_contrast = _zero_none(grads_contrast, weights)
        grads_centroid = _zero_none(grads_centroid, weights)
        # Compose the combined gradient from components to check equality
        composed_grads = []
        for gc, gce in zip(grads_contrast, grads_centroid):
            # scaling same as total_loss: (1-alpha) * recon + alpha * (contrast + centroid)
            composed_grads.append(self.cbcl_weight * (gc + gce))
        # Global norms for each set
        def global_norm(grads):
            sq = [tf.reduce_sum(tf.square(g)) for g in grads]
            return tf.sqrt(tf.add_n(sq) + 1e-12)
        norm_contrast = global_norm(grads_contrast)
        norm_centroid = global_norm(grads_centroid)
        # Print global summary (percent contributions by norm)
        #tf.print("GRAD-NORMS: contrast:", norm_contrast, "centroid:", norm_centroid, '\n')
        #################################################################################
        
        gradients = cbcl_tape.gradient(total_loss, weights)
        gradients, _global_norm_before_clipping = tf.clip_by_global_norm(gradients, self.clip_val)
        grads_and_vars = [(g,w) for g, w in zip(gradients, weights) if g is not None]
        self.optimizer.apply_gradients(grads_and_vars)
        del cbcl_tape
        
        ########### EMA #################       
        old = self.centroids.read_value()
        mask = counts > 1e-6
        mask_exp = tf.expand_dims(mask, -1)
        m = tf.cast(self.centroid_shift_momentum, old.dtype)
        new_centroids = (1.0 - m) * old + m * tf.stop_gradient(curr_batch_centroids)
        new_centroids = tf.where(mask_exp, new_centroids, old)
        self.centroids.assign(new_centroids)
        
        return loss_contrast, loss_centroid, total_loss
           
    def experimental_train(self, dataset:tf.data.Dataset, test_windows, test_labels, batch_size=1024, verbose=False, refit_pca = False, seperate_train=False):
        dataset = dataset.cache()
        if not self.initialized: 
            self.pca = PCA(n_components=3, random_state=42, svd_solver='full')
            self.umap = umap.UMAP(n_components=3, random_state=42)
            self.tsne = TSNE(n_components=3, random_state=42)
            self.compute_initial_centroids(dataset)
            self.initialized = True
            
        train_batches = dataset.shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
        if refit_pca: 
            train_latents = []
            for batch in train_batches:
                train_latents.append(self.encode(batch).numpy())
            train_latents = np.vstack(train_latents)
            self.pca.fit(train_latents)
            self.umap.fit(train_latents)
            
        print('\t Started Training')
        if seperate_train:
            for batch in train_batches:
                loss_mse = self.reconstruction_train_step(batch)
                _, _, loss_cbcl = self.cbcl_train_step(batch)
                print(f'\t\t mse loss: {loss_mse}, cbcl_loss: {loss_cbcl}')
        else:
            for batch in train_batches:
                self.hard_train_step(batch)
        print('\t Finished Training')
                
        if verbose:
            test_batches = test_windows.batch(batch_size).prefetch(tf.data.AUTOTUNE)
            all_train_latents = []
            for batch in train_batches:
                latents = self.encode(batch).numpy()
                all_train_latents.append(latents)
            all_train_latents = np.vstack(all_train_latents)
            
            all_test_latents = []
            for batch in test_batches:
                latents = self.encode(batch).numpy()
                all_test_latents.append(latents)
            all_test_latents = np.vstack(all_test_latents)
            self.visualize_clustering_with_GMM(all_train_latents, all_test_latents, test_labels)


In [16]:
class framework:
    def __init__(self, weights = [0.2, 2, 2]):
        self.weights = weights
    
    def build(self, X_train, expected_anomaly_ratio, frame_length=5, latent_vec_size=10, num_clusters=2, dropout_rate=0.2, lstm_units = 64, regularizer = None, opt_lr=0.001, ema=0.1, clip_val=1):
        assert 0 <= expected_anomaly_ratio <= 1
        self.expected_anomaly_ratio = expected_anomaly_ratio
        self.frame_length = frame_length
        self.num_clusters=num_clusters
        self.timewindows = tf.signal.frame(tf.convert_to_tensor(X_train, dtype=tf.float32), frame_length=frame_length, frame_step=1, axis=0)
        self.dataset = tf.data.Dataset.from_tensor_slices(self.timewindows).batch(2048).cache().prefetch(tf.data.AUTOTUNE)
        self.latent_size = latent_vec_size
        self.model = LSTMAE_CBCL(dimensions=[frame_length, X_train.shape[1]],
                                 lstm_units=lstm_units,
                                 latent_vec_size=latent_vec_size,
                                 weights = self.weights,
                                 num_clusters=num_clusters,
                                 dropout_rate=dropout_rate,
                                 kernel_regularizer=regularizer,
                                 opt_lr=opt_lr, 
                                 clip_val = clip_val,
                                 ema=ema)
        
    def experimental_run_and_test_dirty(self, X_test, y_test, epochs, batch_size, cbcl_weight=1.0, mse_weight=1.0, seperate_train=False, verbose=False):
        training_set = tf.data.Dataset.from_tensor_slices(self.timewindows)
        test_windows = tf.data.Dataset.from_tensor_slices(tf.signal.frame(tf.convert_to_tensor(X_test, dtype=tf.float32), frame_length=self.frame_length, frame_step=1, axis=0))
        test_windows_labels = tf.signal.frame(tf.convert_to_tensor(y_test, dtype=tf.float32), frame_length=self.frame_length, frame_step=1, axis=0)
        reshaped = tf.reshape(test_windows_labels, (tf.shape(test_windows_labels)[0], -1))
        window_has_anom = tf.reduce_any(tf.equal(reshaped, 1), axis=1)   # shape: (num_windows,)
        window_labels = tf.cast(window_has_anom, tf.int32)               # shape: (num_windows,)
        self.model.cbcl_weight = cbcl_weight
        self.model.mse_weight = mse_weight
        
        perf_scores = []
        for e in range(epochs):
            print(f'Epoch {e+1}/{epochs}')
            self.model.experimental_train(training_set, test_windows, window_labels, batch_size=batch_size, verbose=verbose, refit_pca=False, seperate_train=seperate_train)
            
            test_set = test_windows.batch(2048).cache().prefetch(tf.data.AUTOTUNE)
            window_scores = []
            for batch in test_set:
                scores = self.model.predict_anomaly_score(batch)
                window_scores.append(scores)
            window_scores = np.concatenate(window_scores)
            y_preds = (window_scores > np.percentile(window_scores, (1-self.expected_anomaly_ratio)*100))
            
            perf_scores.append([f1_score(window_labels, y_preds), roc_auc_score(window_labels, window_scores), average_precision_score(window_labels, window_scores)])
        
        functions.visualize_scores_through_epochs(perf_scores)

# TODO :
1. ~~Visualize clustering results~~ 
2. Compute grid-search for parameters (including gamma + batch sizing)
3. Frame lengths + extra layers of LSTM 
4. Verify predictions framework (against other machines, verify clustering results / thresholding per clustering)
5. Dropouts + regularizer for LSTM
 

Ideally: want 2 defined clusters for normal / abnormal

Test both (X, X') formation:
- Split batch into 2, map 1:1
- For each batch, form every pairing (exhaustive)
	Test: Train on all pair vs selected pair

~~Average pooling instead of max pooling~~:  
-   ~~Trading higher precision for lower recalls~~
-   Max pooling seems to produce better scorings

Inference: 
1. ~~Threshold based on top anomaly percentile~~ (on pure autoencoder residual, not strong enough)
2. ~~Thresholding based on individual clustering reconstruction threshold.~~ Shows promising results

Average pooling helps not accidentally mislabelling data as abnormal when it is normal since if the data is normal, more windows containing timestamp x is lower score, so average is lower
    If x is abnormal, even on average it would still be high, but the surrounding timestamps wont be polluted by the high window score. However the averaging can also squashes down anomalous value
    This results in higher precision scores while lowering recalls

Max pooling helps separating true anomaly from normal data, but risk polutting nearby timestamps if anomalous timestamp is surrounded by normal timestamp
    This results in higher recall but lowering precisions

In [17]:
alpha = 0.1
lmda = 1
sigma = 0.7

cbcl_weight = 1
mse_weight = 1

ema = 0.1
frame_length=5
num_clusters = 5
latent_dim = 10
lstm_units = 64

opt_lr = 0.001      
weight_decay = 1e-4     # Smaller == Weaker regularization
dropout_rate= 0.2       # Smaller == Weaker regularization
regularizer = keras.regularizers.l2(1e-3)
gradient_clip = 1.0


model_weights = [alpha, lmda, sigma]
global_batch = 256
training_epochs = 50 

# Baseline LSTM

In [ ]:
X_train_full, X_test_full, y_test_full = functions.load_SMD(1,1)
fw = framework(model_weights)
fw.build(X_train_full, expected_anomaly_ratio=0.095, frame_length=frame_length, latent_vec_size=latent_dim, num_clusters=num_clusters, dropout_rate=dropout_rate, lstm_units=lstm_units, regularizer = regularizer, opt_lr=opt_lr, ema=ema)
fw.experimental_run_and_test_dirty(X_test_full, y_test_full, epochs=training_epochs, batch_size=global_batch, cbcl_weight=0, mse_weight=1, verbose=False, seperate_train=True)

Epoch 1/50
